# Day2: Normalization

このノートブックでは、RNA-seqで生の count をそのまま比較しない理由と、正規化の基本を学びます。

このトピックが役立つ場面:
treated サンプルの総read数が control より多いときに、見かけの差と本当の差を分けたい場面です。

このトピックで解く課題:
`IL6` が treated で高いのは、単にたくさん読まれたからなのか、それとも正規化しても高いのかを確かめます。

## なぜ正規化が必要か

サンプルごとに読まれたRNA断片の総量が違うと、同じ発現量でも count が大きく見えたり小さく見えたりします。そこで、まずサンプル全体の量をそろえる処理が必要になります。

In [ ]:
import pandas as pd
import numpy as np

counts = pd.DataFrame({
    "gene": ["TP53", "MYC", "GAPDH", "ACTB", "IL6"],
    "control_1": [120, 300, 5000, 4200, 30],
    "control_2": [100, 280, 5100, 4000, 25],
    "treated_1": [300, 600, 5300, 4100, 200],
    "treated_2": [280, 650, 5200, 4300, 220],
}).set_index("gene")

counts

In [ ]:
library_size = counts.sum(axis=0)
library_size

## CPM でそろえる

ここでは簡単な正規化として CPM (`counts per million`) を使います。各サンプルの count を library size で割り、100万スケールにそろえます。

In [ ]:
cpm = counts.div(library_size, axis=1) * 1_000_000
cpm.round(2)

## log 変換をかける

RNA-seqの値は広い範囲にばらつくので、そのままだと大きい遺伝子に引っ張られやすくなります。`log2(x + 1)` をかけると、分布が扱いやすくなります。

In [ ]:
log_cpm = np.log2(cpm + 1)
log_cpm.round(2)

## 正規化前後で見え方はどう変わるか

生の count では library size の違いが混ざっています。CPM にするとサンプル全体の量をそろえた上で比較でき、さらに log 変換後は PCA やヒートマップに使いやすい値になります。

In [ ]:
comparison = pd.DataFrame({
    "raw_control_1": counts["control_1"],
    "raw_treated_1": counts["treated_1"],
    "log_cpm_control_1": log_cpm["control_1"],
    "log_cpm_treated_1": log_cpm["treated_1"]
})

comparison.round(2)

## ここまでの要点

- 正規化はサンプル全体の量の差を補正するために必要
- CPM は簡単な正規化の一つ
- `log2(x + 1)` は値のスケールを扱いやすくする
- 次はこの正規化済みデータを可視化に使う